# Challenge 2: House Prices - Advanced Regression Techniques

## Huấn luyện mô hình cơ bản (Baseline Model Training)

Các mô hình được huấn luyện trên tập dữ liệu gốc (chưa qua feature engineering), nhằm đánh giá hiệu năng cơ bản ban đầu.

### Khai báo thư viện

In [26]:
import pandas as pd
import numpy as np
import random
import os, sys
from IPython import display


from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

### Tham số thực nghiệm

In [11]:
params = {}

params["exps_dir"] = "../exps"
params["exp_name"] = "challenge2_houseprice_standard"

params["exps_root"] = f'{params["exps_dir"]}/result1_standard'
params["save_dir"] = f'{params["exps_dir"]}/result1_{params["exp_name"]}'

params["data_path"] = f'{params["exps_dir"]}/data/train.xlsx'
params["test_path"] = f'{params["exps_dir"]}/data/test.xlsx'

params["k_fold"] = 10
params["random_state"] = 42

random.seed(params["random_state"])
os.environ['PYTHONHASHSEED'] = str(params["random_state"])
np.random.seed(params["random_state"])

### Nạp dữ liệu

In [12]:
df_train = pd.read_excel(f"{params["data_path"]}")
df_test = pd.read_excel(f"{params["test_path"]}")

In [13]:
df_train.head()

,LotShape,LandContour,LotConfig,Neighborhood,HouseStyle,OverallQual,YearRemodAdd,RoofStyle,Exterior1st,MasVnrArea,...,GrLivArea,FullBath,KitchenQual,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,SalePrice
0,0.750731,0.314667,0.604670,-1.206215,1.026689,0.651479,0.878668,-0.491516,0.743092,0.514104,...,0.370333,0.789741,-0.409369,-0.951226,-0.075117,-0.768736,1.005970,0.103495,0.311725,208500
1,0.750731,0.314667,-0.628316,1.954302,-0.543457,-0.071836,-0.429577,-0.491516,-0.508251,-0.570750,...,-0.482512,0.789741,0.795629,0.600495,1.638498,-0.768736,-0.094619,0.103495,0.311725,181500
2,-1.378933,0.314667,0.604670,-1.206215,1.026689,0.651479,0.830215,-0.491516,0.743092,0.325915,...,0.515013,0.789741,-0.409369,0.600495,1.638498,-0.768736,0.924445,0.103495,0.311725,223500
3,-1.378933,0.314667,-1.861302,-1.039872,1.026689,0.651479,-0.720298,-0.491516,1.055928,-0.570750,...,0.383659,-1.026041,-0.409369,0.600495,-0.931925,1.301075,0.802157,0.942959,1.650307,140000
4,-1.378933,0.314667,-0.628316,0.457215,1.026689,1.374795,0.733308,-0.491516,0.743092,1.366489,...,1.299326,0.789741,-0.409369,0.600495,1.638498,-0.768736,0.883682,0.103495,1.650307,250000


In [14]:
df_test.head()

,LotShape,LandContour,LotConfig,Neighborhood,HouseStyle,OverallQual,YearRemodAdd,RoofStyle,Exterior1st,MasVnrArea,...,CentralAir,GrLivArea,FullBath,KitchenQual,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars
0,0.750731,0.314667,0.604670,-0.041814,-0.543457,-0.795151,-1.156380,-0.491516,0.743092,-0.570750,...,1,-1.179256,-1.026041,0.795629,-0.951226,-0.075117,-0.768736,-0.706058,0.942959,-1.026858
1,-1.378933,0.314667,-1.861302,-0.041814,-0.543457,-0.071836,-1.301740,1.904521,1.055928,0.027027,...,1,-0.354966,-1.026041,-0.409369,-0.951226,-0.075117,-0.768736,-0.828346,0.942959,-1.026858
2,-1.378933,0.314667,0.604670,-0.707186,1.026689,-0.795151,0.636400,-0.491516,0.743092,-0.570750,...,1,0.216136,0.789741,0.795629,0.600495,1.638498,-0.768736,0.761394,-1.575431,0.311725
3,-1.378933,0.314667,0.604670,-0.707186,1.026689,-0.071836,0.636400,-0.491516,0.743092,-0.460051,...,1,0.168544,0.789741,-0.409369,0.600495,-0.931925,-0.768736,0.802157,-1.575431,0.311725
4,-1.378933,-2.512494,0.604670,1.621616,-0.543457,1.374795,0.345679,-0.491516,-1.133923,-0.570750,...,1,-0.448246,0.789741,-0.409369,-0.951226,-0.075117,-0.768736,0.557582,0.103495,0.311725


In [15]:
# Tách X, y
y = df_train["SalePrice"]
X = df_train.drop(columns=["SalePrice"])


print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print("Train X:", X_train.shape)
print("Valid X:", X_valid.shape)
print("Train y:", y_train.shape)
print("Valid y:", y_valid.shape)

X shape: (1460, 27)
y shape: (1460,)
Train X: (1168, 27)
Valid X: (292, 27)
Train y: (1168,)
Valid y: (292,)


In [16]:
kfold = KFold(n_splits=params["k_fold"], shuffle=True, random_state=params["random_state"])
print(f"+ X_train: {len(X_train)}")
for fold, (train_idx, valid_idx) in enumerate(kfold.split(X_train, y_train)):
    print(f'Fold {fold}: ')
    print(f'+ train_idx: {train_idx}')
    print(f'+ valid_idx: {valid_idx}')
    print(f'+ train / valid: {valid_idx}')
    pass

+ X_train: 1168
Fold 0: 
+ train_idx: [   0    1    2 ... 1165 1166 1167]
+ valid_idx: [  23   44   49   51   54   58   70   86  101  107  109  113  128  140
  155  156  158  163  174  192  198  199  209  210  218  231  233  244
  292  296  298  308  319  323  327  328  336  344  352  354  355  362
  376  377  394  405  422  424  425  428  429  435  451  462  467  477
  532  545  548  558  560  561  570  590  596  618  629  643  667  692
  694  695  706  714  731  759  762  807  808  811  816  826  837  849
  884  889  900  905  924  925  926  937  939  946  948  966  967  969
  988 1003 1013 1014 1029 1030 1039 1046 1085 1093 1101 1116 1131 1132
 1137 1154 1159 1160 1161]
+ train / valid: [  23   44   49   51   54   58   70   86  101  107  109  113  128  140
  155  156  158  163  174  192  198  199  209  210  218  231  233  244
  292  296  298  308  319  323  327  328  336  344  352  354  355  362
  376  377  394  405  422  424  425  428  429  435  451  462  467  477
  532  545  548  

### Lựa chọn mô hình mặc định

In [21]:
models = {
    ('Extra Trees', ExtraTreesRegressor(random_state=params["random_state"])),
    ('Random Forest', RandomForestRegressor(random_state=params["random_state"])),
    ('LightGBM', LGBMRegressor(
            random_state=params["random_state"],
            verbose=-1,
            verbosity=-1
    )),
    ('Gradient Boosting', GradientBoostingRegressor(random_state=params["random_state"])),
    ('XGBoost', XGBRegressor(random_state=params["random_state"]))
}

### Huấn luyện và đánh giá từng mô hình

In [29]:

results = []
baseline_results = {}

for name, model in models:
    baseline_results[name] = {"mae": [], "rmse": [], "r2": []}

    print(f'Model {name}:')
    kfold = KFold(n_splits=params["k_fold"], shuffle=True, random_state=params["random_state"])

    for fold, (train_idx, valid_idx) in enumerate(kfold.split(X_train, y_train)):
        X1_train, y1_train = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X1_valid, y1_valid = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

        model.fit(X1_train, y1_train)
        y_pred_valid = model.predict(X1_valid)

        mae = mean_absolute_error(y1_valid, y_pred_valid)
        rmse = np.sqrt(mean_squared_error(y1_valid, y_pred_valid))
        r2 = r2_score(y1_valid, y_pred_valid)

        baseline_results[name]["mae"].append(mae)
        baseline_results[name]["rmse"].append(rmse)
        baseline_results[name]["r2"].append(r2)

    print(f'+ params = {model.get_params()}')
    print(f'+ rmse = {baseline_results[name]["rmse"]}')

    msg = f'+ mean_rmse = {np.mean(baseline_results[name]["rmse"]):.6f} +/- {np.std(baseline_results[name]["rmse"]):.6f}'
    print(msg)
    print()

    results.append([
        name,
        np.mean(baseline_results[name]["mae"]),
        np.mean(baseline_results[name]["rmse"]),
        np.mean(baseline_results[name]["r2"])
    ])

Model Gradient Boosting:
+ params = {'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'squared_error', 'max_depth': 3, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_iter_no_change': None, 'random_state': 42, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}
+ rmse = [np.float64(26825.64985786905), np.float64(23283.475132953834), np.float64(27119.603082380905), np.float64(46906.022254618416), np.float64(23551.351694406425), np.float64(42796.011790129465), np.float64(26925.604114626294), np.float64(23318.833765174157), np.float64(20291.181500140447), np.float64(25471.850271601852)]
+ mean_rmse = 28648.958346 +/- 8397.758726

Model LightGBM:
+ params = {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'lear

In [32]:
df_results = pd.DataFrame(
    results,
    columns=["Model", "Mean_RMSE", "Std_RMSE", "RMSE_List"]
)

df_results = df_results.sort_values(by="Std_RMSE", ascending=True)
display.display(df_results)

,Model,Mean_RMSE,Std_RMSE,RMSE_List
1,LightGBM,18321.935447,28628.559900,0.858377
0,Gradient Boosting,17680.118344,28648.958346,0.854380
2,Random Forest,18251.919125,29338.710468,0.850325
3,XGBoost,19206.038965,30251.949936,0.842209
4,Extra Trees,19307.602507,31005.909020,0.830947


**Nhận xét kết quả thực nghiệm các mô hình**

- Kết quả cho thấy Gradient Boosting và LightGBM là hai mô hình hoạt động tốt nhất, với RMSE trung bình lần lượt là 28,648 và 28,628. LightGBM cũng có độ lệch chuẩn thấp, chứng tỏ mô hình ổn định hơn qua các lần thử.

- Random Forest đạt RMSE 29,338, cao hơn một chút so với hai mô hình tốt nhất, cho thấy hiệu quả ở mức khá nhưng chưa tối ưu bằng các mô hình boosting.

- XGBoost (RMSE 30,251) và Extra Trees (RMSE 31,005) cho kết quả kém hơn, đồng thời có độ dao động lớn hơn giữa các fold -> hai mô hình này chưa phù hợp với dữ liệu hiện tại hoặc cần tối ưu thêm siêu tham số.

=> Gradient Boosting và LightGBM là hai mô hình phù hợp nhất để dự đoán giá nhà trong bài toán này.